# Prism CUDA Validation
Run on Colab with T4/A100 GPU. Compares orthogonal baseline vs Prism (Spectral Imprint + EigenTransfer + 2x LR) on GPT-2 small at real batch/seq scale.

In [ ]:
# Cell 1: Setup
!pip install -q transformers datasets scipy
!nvidia-smi
import torch
print(f'CUDA: {torch.cuda.is_available()}, {torch.cuda.get_device_name(0)}')

In [ ]:
# Cell 2: Clone repo
!git clone https://github.com/realityinspector/prismic-pretraining.git /content/prism
%cd /content/prism

In [ ]:
# Cell 3: Pre-cache models
from transformers import GPT2TokenizerFast, GPT2LMHeadModel
GPT2TokenizerFast.from_pretrained('gpt2')
GPT2LMHeadModel.from_pretrained('gpt2')
print('Models cached')

In [ ]:
# Cell 4: Run orthogonal baseline (5K steps, ~3 hrs on T4, ~45 min on A100)
import sys, os, json, gc, time
import torch
import numpy as np

sys.path.insert(0, '/content/prism')
os.chdir('/content/prism')

from prism.config import TrainConfig, install_signal_handlers
from prism.train import train, _clear_memory
from prism.baselines import make_init_fn

install_signal_handlers()
device = 'cuda'
os.makedirs('prism/results/cuda', exist_ok=True)

# Orthogonal baseline
config_ortho = TrainConfig(
    max_steps=5000,
    eval_steps=[500, 1000, 2000, 3000, 4000, 5000],
    warmup_steps=500,
    log_every=100,
    lr=6.25e-5,
    seed=42,
    device=device,
    batch_size=4,
    grad_accum_steps=16,
    max_length=1024,
    memory_pressure_threshold=5,
)

t0 = time.time()
result_ortho = train(config_ortho, init_fn=make_init_fn('orthogonal'), init_name='small_ortho_s42', verbose=True)
print(f'\nOrtho done in {time.time()-t0:.0f}s')
print(f'Final PPL: {result_ortho["final_ppl"]:.1f}')
print(f'Checkpoints: {result_ortho["checkpoints"]}')

with open('prism/results/cuda/small_ortho_s42.json', 'w') as f:
    json.dump({
        'name': 'small_ortho_s42',
        'final_ppl': result_ortho['final_ppl'],
        'elapsed': result_ortho['elapsed'],
        'checkpoints': {str(k): v for k, v in result_ortho['checkpoints'].items()},
        'seed': 42, 'batch_size': 4, 'grad_accum': 16, 'max_length': 1024,
        'max_steps': 5000, 'lr': 6.25e-5, 'device': device,
    }, f, indent=2)

_clear_memory(device)
gc.collect()
print('Saved ortho result')

In [ ]:
# Cell 5: Run Prism (Spectral Imprint + EigenTransfer + 2x LR)
from prism.pretrained_extract import extract_per_layer, make_hybrid_init_fn

# Load extracted spectra
with open('prism/results/extracted_spectra.json') as f:
    extracted = json.load(f)

# Extract pretrained directions for EigenTransfer
print('Extracting pretrained directions...')
dirs = extract_per_layer('gpt2', include_directions=True, device='cpu')

init_fn = make_hybrid_init_fn(
    extracted['spectra_coeffs'], dirs,
    lam=1.0, align_mode='UV', align_strength=0.5
)
gc.collect()

config_prism = TrainConfig(
    max_steps=5000,
    eval_steps=[500, 1000, 2000, 3000, 4000, 5000],
    warmup_steps=500,
    log_every=100,
    lr=1.25e-4,  # 2x LR
    seed=42,
    device=device,
    batch_size=4,
    grad_accum_steps=16,
    max_length=1024,
    memory_pressure_threshold=5,
)

t0 = time.time()
result_prism = train(config_prism, init_fn=init_fn, init_name='small_prism_s42', verbose=True)
print(f'\nPrism done in {time.time()-t0:.0f}s')
print(f'Final PPL: {result_prism["final_ppl"]:.1f}')
print(f'Checkpoints: {result_prism["checkpoints"]}')

with open('prism/results/cuda/small_prism_s42.json', 'w') as f:
    json.dump({
        'name': 'small_prism_s42',
        'final_ppl': result_prism['final_ppl'],
        'elapsed': result_prism['elapsed'],
        'checkpoints': {str(k): v for k, v in result_prism['checkpoints'].items()},
        'seed': 42, 'batch_size': 4, 'grad_accum': 16, 'max_length': 1024,
        'max_steps': 5000, 'lr': 1.25e-4, 'device': device,
    }, f, indent=2)
print('Saved prism result')

In [ ]:
# Cell 6: Compare results
print('\n' + '='*60)
print('CUDA VALIDATION RESULTS')
print('='*60)
print(f'\nGPU: {torch.cuda.get_device_name(0)}')
print(f'Config: batch 64 (4x16), seq_len 1024, 5000 steps')
print(f'\n{"Step":>6s}  {"Ortho PPL":>10s}  {"Prism PPL":>10s}  {"Ratio":>8s}')
print('-' * 40)

ortho_cp = result_ortho['checkpoints']
prism_cp = result_prism['checkpoints']

for step in [500, 1000, 2000, 3000, 4000, 5000]:
    o = ortho_cp.get(step, None)
    p = prism_cp.get(step, None)
    if o and p:
        ratio = o / p
        print(f'{step:>6d}  {o:>10.1f}  {p:>10.1f}  {ratio:>7.2f}x')

final_ratio = result_ortho['final_ppl'] / result_prism['final_ppl']
print(f'\nFinal: ortho={result_ortho["final_ppl"]:.1f}  prism={result_prism["final_ppl"]:.1f}  ratio={final_ratio:.2f}x')
print(f'\nMPS result was 3.33x. CUDA result is {final_ratio:.2f}x.')
if final_ratio > 2.0:
    print('PRISM ADVANTAGE HOLDS AT SCALE!')
elif final_ratio > 1.3:
    print('Prism advantage shrinks but persists at scale.')
else:
    print('Prism advantage washes out at real scale.')

In [ ]:
# Cell 7: Download results
from google.colab import files
import json
results = {
    'ortho': json.load(open('prism/results/cuda/small_ortho_s42.json')),
    'prism': json.load(open('prism/results/cuda/small_prism_s42.json')),
    'gpu': torch.cuda.get_device_name(0),
}
with open('/content/cuda_validation_results.json', 'w') as f:
    json.dump(results, f, indent=2)
files.download('/content/cuda_validation_results.json')